In [14]:
import os
import numpy as np
import torch as t

from model import Config
from pipeline import OptimizerSpec, Trainer, run_multi_seed
import plots 
import importlib


importlib.reload(plots)

t.manual_seed(0)
np.random.seed(0)

In [6]:
config = Config(
    p=113,
    d_model=128,
    d_mlp=512,
    num_heads=4,
    n_ctx=3,
    act_type='ReLU',
    frac_train=0.3,
    num_epochs=25000,
    seed=0,                    #! cette seed juste poir genreation data, pas pour les seeds multiples
)

specs_adamw = [
    OptimizerSpec('adamw',
                  lr=1e-3,
                  weight_decay=1.0,
                  extra={'betas': (0.9, 0.98)}),
]

SEEDS     = [0, 1, 2, 3, 4]
SAVE_ROOT = 'runs/adamw_baseline'

print(f"Device : {config.device}")
print(f"Specs  : {specs_adamw[0].describe()}")
print(f"Seeds  : {SEEDS}")
print(f"Save   : {SAVE_ROOT}/seedN/")

Device : mps
Specs  : adamw(ALL, lr=0.001, wd=1.0, betas=(0.9, 0.98))
Seeds  : [0, 1, 2, 3, 4]
Save   : runs/adamw_baseline/seedN/


In [7]:

results = {}
already_done = set()
if os.path.exists(SAVE_ROOT):
    for d in sorted(os.listdir(SAVE_ROOT)):
        if d.startswith('seed'):
            seed = int(d.replace('seed', ''))
            results[seed] = Trainer.from_run(f"{SAVE_ROOT}/{d}")
            already_done.add(seed)

missing = [s for s in SEEDS if s not in already_done]
print(f"Already done ({len(already_done)}): {sorted(already_done)}")
print(f"Will run    ({len(missing)}): {missing}")


if missing:
    new_results = run_multi_seed(
        config, specs_adamw,
        seeds=missing,
        label_prefix='adamw_baseline',
        save_root=SAVE_ROOT,
        eval_every=50,
        fourier_every=None,
        warmup_steps=10,
        verbose_every=2000,
        verbose_build=False,
    )
    results.update(new_results)

print()
print(f"Total: {len(results)} seeds disponibles")
for seed in sorted(results):
    h = results[seed].history
    print(f"  seed={seed}: final test_acc={h['test_acc'][-1]:.3f}, "
          f"final train_acc={h['train_acc'][-1]:.3f}")

Already done (0): []
Will run    (5): [0, 1, 2, 3, 4]

=== adamw_baseline seed 0 (1/5) ===
  [adamw_baseline_seed0 seed=0] epoch     0 | train acc 0.009 | test acc 0.009
  [adamw_baseline_seed0 seed=0] epoch  2000 | train acc 1.000 | test acc 0.069
  [adamw_baseline_seed0 seed=0] epoch  4000 | train acc 1.000 | test acc 0.085
  [adamw_baseline_seed0 seed=0] epoch  6000 | train acc 1.000 | test acc 0.115
  [adamw_baseline_seed0 seed=0] epoch  8000 | train acc 1.000 | test acc 0.210
  [adamw_baseline_seed0 seed=0] epoch 10000 | train acc 1.000 | test acc 0.993
  [adamw_baseline_seed0 seed=0] epoch 12000 | train acc 1.000 | test acc 1.000
  [adamw_baseline_seed0 seed=0] epoch 14000 | train acc 1.000 | test acc 1.000
  [adamw_baseline_seed0 seed=0] epoch 16000 | train acc 1.000 | test acc 1.000
  [adamw_baseline_seed0 seed=0] epoch 18000 | train acc 1.000 | test acc 1.000
  [adamw_baseline_seed0 seed=0] epoch 20000 | train acc 1.000 | test acc 1.000
  [adamw_baseline_seed0 seed=0] epoch 22

In [16]:
plots.plot_seeds_overlay(results, title="AdamW baseline — 5 seeds (overlay)")

In [17]:
plots.plot_seeds_band(results, title="AdamW baseline — 5 seeds (band)")